# Geospatial Feature Creation

Create OpenStreetMap proximity features for the prepared Chicago Dwellsy data.

**Data:** real (Dwellsy)

**Inputs:** chicago_prepared_full.csv and local OSM extracts

**Outputs:** geospatial feature tables and figures

**Run after:** 02_data_preparation.ipynb

In [2]:
# No user-adjustable parameters are defined for this notebook.


In [5]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from pyrosm import OSM
import requests


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "libs").is_dir())
sys.path.insert(0, str(ROOT))   # lets the notebook import the shared libs/ package

from libs import paths
from libs.dtypes import standardize_types, get_type_group
from libs.geo import (SEARCH_RADII, EARTH_R, OSM_DIR, BASE_URL, AREAS, CATEGORY_TAGS, download_pbf, distance, generate_features)
from libs.real_data import raw_file


# Import packages


# Load data


In [ ]:
df_real_1 = pd.read_csv(raw_file("chicago"))
df_real_2 = pd.read_csv(raw_file("dc"))

print("Shape of df_real_1:", df_real_1.shape)
print("Shape of df_real_2:", df_real_2.shape)


Chicago shape: (545312, 47)
Washington DC shape: (430605, 47)


In [8]:
df_raw = pd.concat([df_real_1, df_real_2], ignore_index=True)
print("Shape of combined df_raw:", df_raw.shape)
df_raw.head()


Shape of combined df_raw: (975917, 47)


,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,...,ss_raw_secondary_designator,ss_raw_secondary_number,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type
0,133822-8163780,133822,8163780.0,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28.0,28,...,Apt,501,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230.0,Apartment
1,129521-8112041,129521,8112041.0,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296.0,218,...,Apt,807,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919.0,Apartment
2,123813-8066896,123813,8066896.0,14582015,16668829,279329,21st Street Lofts,3L Living,137.0,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
3,123813-8066896,123813,8066896.0,14582015,19878436,279329,21st Street Lofts,3L Living,137.0,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
4,130630-8121250,130630,8121250.0,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54.0,54,...,Apt,206,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523.0,Apartment


# Create geospatial features


In [9]:
df_geo_input = df_raw.copy()
df_geo_input.head()


,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,...,ss_raw_secondary_designator,ss_raw_secondary_number,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type
0,133822-8163780,133822,8163780.0,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28.0,28,...,Apt,501,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230.0,Apartment
1,129521-8112041,129521,8112041.0,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296.0,218,...,Apt,807,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919.0,Apartment
2,123813-8066896,123813,8066896.0,14582015,16668829,279329,21st Street Lofts,3L Living,137.0,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
3,123813-8066896,123813,8066896.0,14582015,19878436,279329,21st Street Lofts,3L Living,137.0,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
4,130630-8121250,130630,8121250.0,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54.0,54,...,Apt,206,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523.0,Apartment


In [10]:
print("Initial shape:", df_geo_input.shape)
df_geo_input = df_geo_input[['dwellsy_address_id', 'address_city', 'latitude', 'longitude']].drop_duplicates()
print("Final shape:", df_geo_input.shape)
df_geo_input.head()


Initial shape: (975917, 47)
Final shape: (263926, 4)


,dwellsy_address_id,address_city,latitude,longitude
0,133822-8163780,Chicago,41.994980,-87.656840
1,129521-8112041,Chicago,41.895200,-87.653970
2,123813-8066896,Chicago,41.853916,-87.624672
4,130630-8121250,Chicago,41.925404,-87.708099
7,118522-7969432,Evanston,42.041416,-87.683342


In [11]:
# Configuration

OSM_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:

# Path local (sin descargar)
for area in AREAS.values():
    area["path"] = OSM_DIR / area["file"]

for area in AREAS.values():
    download_pbf(area["file"])


In [14]:
# Extract POI tags
OSM_FILTER = {}

for tags in CATEGORY_TAGS.values():
    for key, values in tags.items():
        OSM_FILTER.setdefault(key, set()).update(values)

OSM_FILTER = {
    key: sorted(values)
    for key, values in OSM_FILTER.items()
}


# Calculate geographic distances


# Generate geospatial features


In [15]:
# Initialize feature DataFrame
feature_columns = []

for category in CATEGORY_TAGS:
    for radius in SEARCH_RADII:
        feature_columns.append(f"{category}_{radius}m")

    feature_columns.append(f"nearest_{category}_m")


features_df = pd.DataFrame(
    np.nan,
    index=df_geo_input.index,
    columns=feature_columns
)


# Process each geographic area
for area_name, area in AREAS.items():
    print(f"\nProcessing area: {area_name}")

    west, south, east, north = area["bbox"]

    area_mask = (
        df_geo_input["longitude"].between(west, east)
        & df_geo_input["latitude"].between(south, north)
    )

    area_locations = df_geo_input.loc[area_mask]

    if area_locations.empty:
        print("No locations found in this area.")
        continue

    print(f"Locations to process: {len(area_locations)}")

    osm = OSM(
        str(area["path"]),
        bounding_box=area["bbox"],
        complete_relations=True
    )

    pois = osm.get_pois(custom_filter=OSM_FILTER)

    print(f"POIs extracted: {len(pois)}")

    pois = pois[pois.geometry.notna()].copy()

    # Project geometries to a suitable coordinate system
    pois_projected = pois.to_crs(pois.estimate_utm_crs())

    # Calculate centroids
    poi_centroids = pois_projected.geometry.centroid

    # Convert centroids back to geographic coordinates
    poi_centroids = poi_centroids.to_crs(pois.crs)

    pois["poi_lat"] = poi_centroids.y
    pois["poi_lon"] = poi_centroids.x

    category_pois = {}

    for category, tags in CATEGORY_TAGS.items():
        category_mask = pd.Series(False, index=pois.index)

        for key, values in tags.items():
            if key in pois.columns:
                category_mask |= pois[key].isin(values)

        category_data = pois.loc[category_mask]

        category_pois[category] = category_data[
            ["poi_lat", "poi_lon"]
        ].to_numpy()

    for idx, row in area_locations.iterrows():
        location_features = generate_features(
            row["latitude"],
            row["longitude"],
            category_pois
        )

        for feature, value in location_features.items():
            features_df.loc[idx, feature] = value

    print(f"Finished processing: {area_name}")


# Combine original data and geospatial features
df_geo_feats = pd.concat(
    [df_geo_input, features_df],
    axis=1
)

print("Final shape:", df_geo_feats.shape)



Processing area: chicago
Locations to process: 143911


Reading illinois-latest.osm.pbf: decoding the blocks:  12%|#2        | 00:02<00:14

POIs extracted: 48445
Finished processing: chicago

Processing area: dfw
No locations found in this area.

Processing area: dc
Locations to process: 102046
POIs extracted: 6680
Finished processing: dc
Final shape: (263926, 44)


In [16]:
df_geo_feats


,dwellsy_address_id,address_city,latitude,longitude,hospitals_500m,hospitals_1000m,hospitals_1500m,nearest_hospitals_m,pharmacies_500m,pharmacies_1000m,...,atms_1500m,nearest_atms_m,parks_500m,parks_1000m,parks_1500m,nearest_parks_m,public_transport_500m,public_transport_1000m,public_transport_1500m,nearest_public_transport_m
0,133822-8163780,Chicago,41.994980,-87.656840,0.0,0.0,0.0,2700.075732,2.0,3.0,...,0.0,1841.005882,5.0,15.0,33.0,101.759048,18.0,43.0,105.0,116.271259
1,129521-8112041,Chicago,41.895200,-87.653970,0.0,0.0,0.0,2587.437341,1.0,1.0,...,2.0,1258.049233,2.0,19.0,49.0,260.574322,20.0,66.0,144.0,44.990152
2,123813-8066896,Chicago,41.853916,-87.624672,0.0,1.0,1.0,834.926064,0.0,2.0,...,2.0,775.398387,8.0,31.0,49.0,362.245455,33.0,85.0,141.0,58.123987
4,130630-8121250,Chicago,41.925404,-87.708099,0.0,0.0,0.0,2743.833116,1.0,2.0,...,0.0,1670.464777,3.0,15.0,42.0,267.537048,26.0,91.0,157.0,66.105135
7,118522-7969432,Evanston,42.041416,-87.683342,0.0,0.0,0.0,1878.533853,0.0,1.0,...,4.0,656.425910,6.0,24.0,53.0,284.336656,19.0,69.0,117.0,134.500703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
975900,9008429-19071092960,Frederick,39.377945,-77.428230,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
975904,6378724,Fredericksburg,38.276493,-77.532013,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
975906,10357052-60114508633,Washington,38.938839,-77.028557,0.0,0.0,0.0,1550.559044,1.0,2.0,...,7.0,411.730265,12.0,40.0,78.0,179.065850,21.0,75.0,152.0,321.624789
975907,6960124-4574933597,Rockville,39.059666,-77.125900,0.0,0.0,0.0,13752.925990,0.0,0.0,...,0.0,11948.162687,0.0,0.0,0.0,10420.004771,0.0,0.0,0.0,10343.969796


## Export Geospatial Features for Downstream Modeling

The geospatial processing generated 40 proximity-based features for unique property locations in Chicago and Washington, D.C.

These features include counts of nearby points of interest within predefined distance thresholds and distances to the nearest points of interest by category.

We export the resulting feature table to the processed data directory so that subsequent modeling notebooks can reuse the computed features without repeating the OpenStreetMap processing.

The exported dataset preserves property location identifiers and coordinates to support integration with the original rental listings.


In [17]:
# Export geospatial features
output_dir = paths.PROCESSED
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "geospatial_features.csv"

df_geo_feats.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", df_geo_feats.shape)


Saved to: ../../../data/processed/geospatial_features.csv
Shape: (263926, 44)
